# Слои CNN: пулинг, поле восприятия и нормализации

Дополнительные примеры: фиксированная и адаптивная голова, границы областей AdaptiveAvgPool, перекрытие окон, стоимость операций, расстояние между центрами полей восприятия, LayerNorm и GroupNorm.

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from support.week11_utils import show_table, draw_matrices
from support.lesson11_visuals import setup_style

setup_style()
pooling_input = torch.tensor([[[[-3., 1., 2., 0.], [4., -2., 1., 3.],
                                [1., 5., -1., 2.], [0., 2., 6., -4.]]]])

### Глобальное усреднение

Global average pooling усредняет **всё H×W каждого канала отдельно**:

$$z_{n,c}=\frac1{HW}\sum_{h=0}^{H-1}\sum_{w=0}^{W-1}X_{n,c,h,w}.$$

Размер N×C×H×W превращается в N×C×1×1. В PyTorch это `AdaptiveAvgPool2d((1,1))`; после Flatten остаётся N×C.

In [ ]:
global_average = nn.AdaptiveAvgPool2d((1, 1))(pooling_input)
manual_global_average = pooling_input.mean(dim=(2, 3), keepdim=True)
assert torch.allclose(global_average, manual_global_average)
print("До:", tuple(pooling_input.shape), "-> global average:", tuple(global_average.shape))
print("Одно число на канал:", global_average.detach().flatten().tolist())

### Зачем фиксировать размер перед классификатором

Фотографии могут иметь разные H и W. Свёрточные веса не закреплены за конкретным размером изображения: та же свёрточная часть обрабатывает разные пространственные размеры, если ядра и шаги позволяют получить непустой выход. Но после Flatten число признаков равно **C·H·W**, а `Linear(M,10)` требует ровно **M входных признаков**.

Например, при C=8 карта 16×16 даёт 2048 признаков, а 20×28 — 4480. Линейный слой, созданный для 2048, не примет 4480. Есть два распространённых решения:

1. **Resize изображения до фиксированного H×W** перед свёртками. При одной архитектуре на вход головы придёт одно число признаков.
2. **AdaptiveAvgPool карт признаков перед головой**. Свёртки получают исходный размер; pooling выдаёт заданный размер. Для выхода 4×4 всегда M=16C, для global average — M=C.

Можно также усреднить само изображение до фиксированного размера перед свёртками. Это ещё один способ предобработки: он усредняет пиксели, тогда как pooling перед головой усредняет уже вычисленные признаки. Эти операции происходят на разных этапах сети и в общем случае не взаимозаменяемы.

Разные размеры допустимы в разных вызовах модели. Обычный тензор батча N×C×H×W всё равно требует общих H и W: изображения группируют по размеру, дополняют или обрабатывают отдельно. На семинаре сравним три подхода с группировкой по размеру.

In [ ]:
# Одна свёрточная часть, но две разные головы.
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(31)
    demo_features = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU())
    fixed_head = nn.Sequential(nn.Flatten(), nn.Linear(8 * 16 * 16, 10))
    flexible_head = nn.Sequential(nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(), nn.Linear(8 * 4 * 4, 10))

shape_rows = []
for height, width in [(16, 16), (20, 28), (32, 24)]:
    image_batch = torch.zeros(2, 3, height, width)
    features = demo_features(image_batch)
    count = features[0].numel()
    status = "подходит" if count == 8 * 16 * 16 else "число признаков не совпало"
    if count == 8 * 16 * 16:
        assert fixed_head(features).shape == (2, 10)
    flexible_output = flexible_head(features)
    assert flexible_output.shape == (2, 10)
    shape_rows.append({"изображение": f"2x3x{height}x{width}", "после свёртки": tuple(features.shape),
                       "Flatten без pooling": count, "Linear(2048,10)": status,
                       "после AdaptiveAvgPool(4)": 8 * 4 * 4, "выход гибкой головы": tuple(flexible_output.shape)})
show_table(pd.DataFrame(shape_rows))

### Как AdaptiveAvgPool2d строит области

Задаётся **размер выхода**, а не размер ядра. Пусть вход имеет высоту H, выход — P. Для выходной строки i, 0≤i<P, PyTorch вычисляет границы входной области:

$$s_i=\left\lfloor\frac{iH}{P}\right\rfloor,\qquad
e_i=\left\lceil\frac{(i+1)H}{P}\right\rceil.$$

Начало включено, конец не включён: используются строки `s_i:e_i`. Аналогично для ширины W и выходной ширины Q:

$$t_j=\left\lfloor\frac{jW}{Q}\right\rfloor,\qquad
u_j=\left\lceil\frac{(j+1)W}{Q}\right\rceil.$$

Затем считают среднее по полученному прямоугольнику, отдельно для n и c:

$$Y_{n,c,i,j}=\frac{1}{(e_i-s_i)(u_j-t_j)}
\sum_{h=s_i}^{e_i-1}\sum_{w=t_j}^{u_j-1}X_{n,c,h,w}.$$

Слой не обучает границы и не решает оптимизационную задачу для kernel_size/stride: границы получаются из этих формул. Они приведены в [реализации PyTorch, start_index/end_index](https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/native/AdaptivePooling.h).

При H=8 и P=3 получаем `[0:3)`, `[2:6)`, `[5:8)`: размеры 3, 4, 3, строки 2 и 5 попали в две области. Поэтому в общем случае нет одного фиксированного ядра и шага. Если H делится на P, области по высоте имеют размер и шаг H/P без перекрытия; аналогично по ширине. При 32×32→4×4 это обычный AvgPool с kernel_size=stride=8.

Целевой выход может быть больше входа. Тогда входные позиции используются повторно: это не создаёт новых деталей изображения и не является обучаемым увеличением разрешения.

In [ ]:
# Границы по высоте 8 -> 3: целочисленная запись floor и ceil.
height, output_height = 8, 3
bin_rows = []
row_windows = []
for index in range(output_height):
    start = index * height // output_height
    end = ((index + 1) * height + output_height - 1) // output_height
    bin_rows.append({"строка выхода i": index, "начало s_i": start, "конец e_i (не включён)": end,
                     "число строк": end - start})
    row_windows.append(np.arange(height)[start:end].reshape(1, -1))
show_table(pd.DataFrame(bin_rows))
draw_matrices([np.arange(height).reshape(1, -1)] + row_windows,
              ["Индексы входа", "i=0: [0:3)", "i=1: [2:6)", "i=2: [5:8)"])

adaptive_input = torch.arange(40., dtype=torch.float64).reshape(1, 1, 8, 5)
adaptive_output = nn.AdaptiveAvgPool2d((3, 2))(adaptive_input)
manual_adaptive = torch.empty_like(adaptive_output)
for row in range(3):
    start_h = row * 8 // 3
    end_h = ((row + 1) * 8 + 3 - 1) // 3
    for column in range(2):
        start_w = column * 5 // 2
        end_w = ((column + 1) * 5 + 2 - 1) // 2
        region = adaptive_input[:, :, start_h:end_h, start_w:end_w]
        manual_adaptive[:, :, row, column] = region.mean(dim=(2, 3))
assert torch.allclose(adaptive_output, manual_adaptive)
from support.lesson11_visuals import plot_adaptive_regions
plot_adaptive_regions(adaptive_input.numpy()[0, 0], adaptive_output.numpy()[0, 0])

### Сколько это стоит по времени

Для одной карты выполняется примерно

$$\sum_{i=0}^{P-1}(e_i-s_i)\;\sum_{j=0}^{Q-1}(u_j-t_j)$$

чтений элементов для суммирования; для батча и каналов — в NC раз больше. При делимых размерах области не перекрываются и сумма равна HW, как у обычного усреднения неперекрывающихся окон. В примере 8×5→3×2 суммы длин равны 10 и 6: получается 60 участий входных элементов вместо 40 разных пикселей.

Это число операций, а не готовый коэффициент задержки. Время также зависит от формы тензора, устройства, реализации ядра и накладных расходов запуска. AdaptiveAvgPool не обязан всегда работать медленнее AvgPool или MaxPool. Ниже измеряем **одинаковые входы и размер выхода 4×4**; обычный AvgPool в этих случаях даёт те же значения. MaxPool включён для сравнения стоимости, но вычисляет другой результат. Замер относится к forward, без загрузки данных и backward.

In [ ]:
from support.week11_utils import benchmark_inference, choose_device
from support.lesson11_visuals import plot_pool_timings

pool_devices = [torch.device("cpu")]
accelerator = choose_device()
if accelerator.type != "cpu":
    pool_devices.append(accelerator)

pool_timing_rows = []
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(72)
    for shape in [(1, 16, 32, 32), (32, 32, 64, 64), (32, 32, 128, 128)]:
        cpu_example = torch.randn(shape)
        kernel = shape[-1] // 4
        fixed_average = nn.AvgPool2d(kernel, stride=kernel)
        adaptive_average = nn.AdaptiveAvgPool2d((4, 4))
        assert torch.allclose(fixed_average(cpu_example), adaptive_average(cpu_example), atol=1e-6)
        for device in pool_devices:
            example = cpu_example.to(device)
            measurements = []
            for name, operation in [("AvgPool", fixed_average), ("AdaptiveAvgPool", adaptive_average),
                                    ("MaxPool", nn.MaxPool2d(kernel, stride=kernel))]:
                measured = benchmark_inference(operation.to(device), example, device, repeats=50, warmup=15)
                measurements.append({"вход": "x".join(map(str, shape)), "выход": "x".join(map(str, (*shape[:2], 4, 4))),
                                     "устройство": str(device), "операция": name,
                                     "медиана, мкс": measured["latency_ms"] * 1000})
            baseline = measurements[0]["медиана, мкс"]
            for row in measurements:
                row["отношение к AvgPool"] = row["медиана, мкс"] / baseline
                pool_timing_rows.append(row)
pool_timing_table = pd.DataFrame(pool_timing_rows)
show_table(pool_timing_table)
plot_pool_timings(pool_timing_table);

## Поле восприятия и перенос признаков

Поле восприятия — область исходного изображения, которая может влиять на один
элемент карты признаков. Пусть $r$ — ее размер, $j$ — расстояние между соседними
элементами карты в координатах входа. Изначально $r_0=1,j_0=1$.

Слой с ядром $k$, dilation $d$ объединяет $d(k-1)+1$ положений. К прежнему полю
добавляется $d(k-1)$ промежутков длины $j$:

$$r_{new}=r+d(k-1)j,\qquad j_{new}=sj.$$

### Что означает j: расстояние между центрами

$j$ измеряется **в пикселях исходного изображения**. Когда на текущей выходной карте переходим от элемента u к соседнему u+1, центр его поля восприятия сдвигается на j исходных пикселей. Это не размер поля r и не только stride последнего слоя.

Входные пиксели стоят через один: j=1. Первый pooling со stride=2 даёт j=2; второй — j=4. Между ними Conv со stride=1 увеличивает r, но оставляет j прежним. В общем случае $j_l=s_lj_{l-1}=\prod_{q=1}^{l}s_q$.

В цепочке ниже последний Conv имеет stride=1, но соседние выходы смотрят на центры, отстоящие на 4 пикселя исходной картинки. Каждый из них при этом имеет поле размера 18.

In [ ]:
receptive_field = 1
jump = 1
rows = []
for name, kernel, stride in [("conv1", 3, 1), ("pool1", 2, 2), ("conv2", 3, 1),
                             ("pool2", 2, 2), ("conv3", 3, 1)]:
    receptive_field = receptive_field + (kernel - 1) * jump
    jump = stride * jump
    rows.append({"слой": name, "поле восприятия": receptive_field, "расстояние центров, пиксели исходного входа": jump})
show_table(pd.DataFrame(rows))

from support.lesson11_visuals import plot_receptive_jump
plot_receptive_jump(receptive_field, jump);

### Какие оси нормирует BatchNorm

На схеме **Batch Norm** синим выделены значения, для которых вычисляют одну пару среднего и дисперсии. Ось N — объекты батча, C — каналы, (H,W) — пространственные позиции.

<img src="assets/normalization_dropout/batch_norm_axes.png" width="360" alt="Batch Norm: один канал, все объекты N и пространственные позиции H,W">

Источник: [Wu, He. Group Normalization (2018), рис. 2, Batch Norm](https://arxiv.org/abs/1803.08494).

Синяя плоскость проходит через **все объекты N и все пространственные позиции H,W одного канала C**. Для входа N×C×H×W усреднение идёт по (N,H,W), то есть `dim=(0,2,3)`. Каналы не смешиваются. Каждая статистика имеет форму 1×C×1×1 и распространяется на исходный тензор по правилам broadcasting.

Ниже разбираем небольшой блок `Conv2d → BatchNorm2d → ReLU`. У свёртки два выходных фильтра: каждый создаёт один канал. BatchNorm получает выход свёртки **до ReLU** и сохраняет его форму. N=3 — три объекта батча, C=2 — два канала, H=2 и W=3 — размер каждой карты.

Для числового примера берём свёртку 1×1 с двумя входными и двумя выходными каналами: первый канал копируется, второй умножается на 2 и получает сдвиг +10. Это выбранные для демонстрации веса, а не результат обучения. У BatchNorm в этом примере отключены обучаемый масштаб и сдвиг (`affine=False`), поэтому рисунок показывает именно нормированные значения.

In [ ]:
from support.lesson11_visuals import plot_bn_block

block_input = torch.arange(36, dtype=torch.float64).reshape(3, 2, 2, 3)
conv_before_bn = nn.Conv2d(2, 2, kernel_size=1, bias=True).double()
with torch.no_grad():
    conv_before_bn.weight.zero_()
    conv_before_bn.weight[0, 0, 0, 0] = 1
    conv_before_bn.weight[1, 1, 0, 0] = 2
    conv_before_bn.bias.copy_(torch.tensor([0., 10.], dtype=torch.float64))
    normalization_input = conv_before_bn(block_input)

normalization_block = nn.Sequential(
    conv_before_bn,
    nn.BatchNorm2d(2, affine=False, track_running_stats=False).double(),
    nn.ReLU(),
)
block_output = normalization_block(block_input)
assert block_output.shape == block_input.shape
plot_bn_block(normalization_input.numpy());

In [ ]:
from support.lesson11_visuals import plot_bn_tensor

channel_mean = normalization_input.mean(dim=(0, 2, 3), keepdim=True)
channel_variance = normalization_input.var(dim=(0, 2, 3), unbiased=False, keepdim=True)
normalized_manual = (normalization_input - channel_mean) / torch.sqrt(channel_variance + 1e-5)

batch_norm_demo = nn.BatchNorm2d(2, affine=False, track_running_stats=False).double()
bn_values = batch_norm_demo(normalization_input)
assert torch.allclose(bn_values, normalized_manual)
assert channel_mean.shape == (1, 2, 1, 1)
assert torch.allclose(bn_values.mean(dim=(0, 2, 3)), torch.zeros(2, dtype=torch.float64), atol=1e-10)

assert torch.allclose(block_output, torch.relu(bn_values))

print("Статистики:", tuple(channel_mean.shape), "-> вход:", tuple(normalization_input.shape))
plot_bn_tensor(normalization_input.numpy(), bn_values.numpy());

В этом числовом примере у BatchNorm накопление статистик выключено для проверки формулы. В обычной CNN используют `track_running_stats=True`: в train считают статистики батча и обновляют накопленные, в eval используют накопленные.

## LayerNorm и GroupNorm

Сравним способы объединять элементы тензора. Для BatchNorm нормируются значения одного канала по батчу и пространству; для LayerNorm - все каналы и пространство одного объекта; для GroupNorm - группа каналов и пространство одного объекта. В числовом примере BatchNorm отключаем обучаемый масштаб, сдвиг и накопленные статистики, чтобы сравнить именно оси нормирования.

### LayerNorm: все каналы одного объекта

У `LayerNorm((C,H,W))` другая группа: **все C,H,W одного объекта**, без соседей по N. Для этого варианта получаем N отдельных средних, а не C. В других архитектурах `normalized_shape` может задавать другой набор последних осей. LayerNorm не хранит running mean/variance.

В train и eval LayerNorm использует статистики текущего объекта и одинаковые вычисления.

In [ ]:
layer_norm_demo = nn.LayerNorm((2, 2, 3), elementwise_affine=False).double()
ln_values = layer_norm_demo(normalization_input)
object_mean = ln_values.mean(dim=(1, 2, 3))
assert torch.allclose(object_mean, torch.zeros(3, dtype=torch.float64), atol=1e-10)
print("После BatchNorm, среднее по N,H,W каждого C:", bn_values.mean(dim=(0, 2, 3)))
print("После LayerNorm, среднее по C,H,W каждого N:", object_mean)

### GroupNorm: группы каналов внутри одного объекта

На следующей схеме **Group Norm** синим объединены несколько каналов и все их пространственные позиции **одного объекта**. По оси N группа не продолжается: статистики разных объектов независимы.

<img src="assets/normalization_dropout/group_norm_axes.png" width="360" alt="Group Norm: группа каналов C и все H,W одного объекта, без объединения по N">

Источник: [Wu, He. Group Normalization (2018), схема авторов](https://ppwwyyxx.com/publication/group-norm/).

В `nn.GroupNorm(num_groups=G, num_channels=C)` число групп G задают при создании слоя. C должно делиться на G. Каналы в их текущем порядке разбивают на G равных последовательных групп; это не кластеризация и не случайная перестановка. Веса предыдущей свёртки обучаются, но правило разбиения не обучается.

Например, C=8 и G=4 дают группы **[0,1], [2,3], [4,5], [6,7]**. В каждой группе C/G=2 канала. При H=2, W=3 для одного объекта и одной группы собирают 2·2·3=12 чисел.

Обозначим набор каналов группы g через $\mathcal C_g$, а число значений в ней через $K=(C/G)HW$. Для объекта n вычисляют:

$$\mu_{n,g}=\frac1K\sum_{c\in\mathcal C_g}\sum_{h=0}^{H-1}\sum_{w=0}^{W-1}x_{n,c,h,w},$$
$$v_{n,g}=\frac1K\sum_{c\in\mathcal C_g}\sum_{h=0}^{H-1}\sum_{w=0}^{W-1}(x_{n,c,h,w}-\mu_{n,g})^2,$$
$$\hat x_{n,c,h,w}=\frac{x_{n,c,h,w}-\mu_{n,g}}{\sqrt{v_{n,g}+\varepsilon}},\qquad c\in\mathcal C_g,$$
$$y_{n,c,h,w}=\gamma_c\hat x_{n,c,h,w}+\beta_c.$$

Сначала внутри группы вычитают её среднее, затем делят на её стандартное отклонение. Обучаемые $\gamma_c,\beta_c$ задаются **на каждый канал**, хотя среднее и дисперсия общие для группы. Получается N·G пар статистик и 2C обучаемых чисел.

Для вычислений удобно представить вход как N×G×(C/G)×H×W и усреднить последние три оси: `dim=(2,3,4)`. Для нашего примера это 2×4×2×2×3; форма статистик — 2×4×1×1×1.

У GroupNorm **в train и eval используются статистики текущего объекта**. Накопленных `running_mean` и `running_var` нет. Изменение другого объекта батча не меняет результат первого. При G=1 в группе все каналы одного объекта; при G=C в каждой группе один канал, нормирование идёт только по его пространственным позициям. [Соглашения PyTorch GroupNorm](https://docs.pytorch.org/docs/2.15/generated/torch.nn.GroupNorm.html).

In [ ]:
from support.lesson11_visuals import plot_group_channels

group_input = torch.arange(96, dtype=torch.float64).reshape(2, 8, 2, 3)
group_count = 4
grouped = group_input.reshape(2, group_count, 2, 2, 3)
group_mean = grouped.mean(dim=(2, 3, 4), keepdim=True)
group_variance = grouped.var(dim=(2, 3, 4), unbiased=False, keepdim=True)
group_manual = (grouped - group_mean) / torch.sqrt(group_variance + 1e-5)
group_manual = group_manual.reshape_as(group_input)

group_norm = nn.GroupNorm(num_groups=4, num_channels=8, affine=False).double()
group_norm.train()
group_values = group_norm(group_input)
assert torch.allclose(group_values, group_manual)
assert group_mean.shape == (2, 4, 1, 1, 1)

changed_batch = group_input.clone()
changed_batch[1] += 1000
assert torch.allclose(group_norm(changed_batch)[0], group_values[0])
group_norm.eval()
assert torch.allclose(group_norm(group_input), group_values)

plot_group_channels(group_input.numpy(), group_count)
show_table(pd.DataFrame({
    "объект n": np.repeat(np.arange(2), 4),
    "группа g": np.tile(np.arange(4), 2),
    "каналы": ["0, 1", "2, 3", "4, 5", "6, 7"] * 2,
    "mu группы": group_mean.flatten().numpy(),
    "v группы": group_variance.flatten().numpy(),
}))
print("Формула совпала с GroupNorm; другой объект не повлиял на первый; train и eval совпали.")